In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install transformers==4.37.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 22.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 54.6 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.42.4
    Uninstalling transformers-4.42.4:
      Successfully uninstalled transformers-4.42.4


In [3]:
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModel
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load data
with open('/content/drive/MyDrive/train_pos_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_pos_content = file.readlines()

with open('/content/drive/MyDrive/train_neg_full_cleaned.txt', 'r', encoding='utf-8') as file:
    train_neg_content = file.readlines()

# Create DataFrame
train_pos = pd.DataFrame(train_pos_content, columns=['tweet'])
train_pos['label'] = 1
train_neg = pd.DataFrame(train_neg_content, columns=['tweet'])
train_neg['label'] = 0
train = pd.concat([train_pos, train_neg], ignore_index=True)

# Split data
tweets = train['tweet']
labels = train['label']
train_tweets, val_tweets, train_labels, val_labels = train_test_split(tweets, labels, test_size=0.1, random_state=42)

# Name of the RoBERTa
roberta = "cardiffnlp/twitter-roberta-base-sentiment"

# Tokenize
tokenizer = AutoTokenizer.from_pretrained(roberta)

max_len = 40
X_train_encoded = tokenizer(train_tweets.tolist(),
                            padding=True,
                            truncation=True,
                            max_length=max_len,
                            return_tensors='tf')

X_val_encoded = tokenizer(val_tweets.tolist(),
                          padding=True,
                          truncation=True,
                          max_length=max_len,
                          return_tensors='tf')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [4]:
print(X_train_encoded)

{'input_ids': <tf.Tensor: shape=(2250000, 40), dtype=int32, numpy=
array([[   0,  405,   16, ...,    1,    1,    1],
       [   0,  118, 2649, ...,    1,    1,    1],
       [   0, 1322,   47, ...,    1,    1,    1],
       ...,
       [   0, 1322,   47, ...,    1,    1,    1],
       [   0, 1178, 6777, ...,    1,    1,    1],
       [   0, 6025,   16, ...,    1,    1,    1]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(2250000, 40), dtype=int32, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int32)>}


In [5]:
# Load BERT model without the classification head
bert_model = TFAutoModel.from_pretrained(roberta)
#bert_model.trainable = False  # Make BERT non-trainable

input_ids = tf.keras.Input(shape=(max_len,), dtype='int32')
attention_mask = tf.keras.Input(shape=(max_len,), dtype='int32')

bert_outputs = bert_model([input_ids, attention_mask])
sequence_output = bert_outputs.last_hidden_state

tf_model.h5:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some layers from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment were not used when initializing TFRobertaModel: ['classifier']
- This IS expected if you are initializing TFRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFRobertaModel were initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaModel for predictions without further training.


## MODEL2

In [6]:
conv_filter_size = 64
conv_kernel_size1 = 2
conv_kernel_size2 = 4
lstm_unit = 256
num_heads = 8
key_dim = (int)(conv_filter_size / num_heads)
dropout_rate = 0.2


# Bidirectional LSTM layer
bilstm_layer = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_unit, dropout = 0.2, recurrent_dropout = 0.2, return_sequences=True))(sequence_output)

# Convolutional layer
conv_layer1 = tf.keras.layers.Conv1D(filters=conv_filter_size, kernel_size=conv_kernel_size1, padding='same', activation='relu', kernel_initializer="he_uniform")(bilstm_layer)
conv_layer2 = tf.keras.layers.Conv1D(filters=conv_filter_size, kernel_size=conv_kernel_size2, padding='same', activation='relu', kernel_initializer="he_uniform")(bilstm_layer)

conv_layer = tf.keras.layers.Concatenate()([conv_layer1, conv_layer2])

# Multi-Head Attention layer
multi_head_attention = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(conv_layer, conv_layer)
multi_head_attention = tf.keras.layers.Dropout(dropout_rate)(multi_head_attention)

global_pool = tf.keras.layers.Concatenate()([tf.keras.layers.GlobalAveragePooling1D()(multi_head_attention), tf.keras.layers.GlobalMaxPooling1D()(multi_head_attention)])

# dense layer
global_pool = tf.keras.layers.Dense(64, activation = "relu")(global_pool)
output = tf.keras.layers.Dense(1, activation='sigmoid')(global_pool)

model = tf.keras.models.Model(inputs=[input_ids, attention_mask], outputs=output)

In [7]:
import tensorflow as tf
print(tf.__version__)

2.15.0


In [8]:
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate = 2e-5), loss='binary_crossentropy', metrics=['accuracy'])
#model.compile(optimizer = 'adam', loss='binary_crossentropy', metrics=['accuracy'])

# uncomment to load model parameter from trained epochs
#checkpoint_path = '/content/drive/MyDrive/MODEL2_RoBERTa/cp-0002.ckpt'
#model.load_weights(checkpoint_path)

print(model.summary())
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 40)]                 0         []                            
                                                                                                  
 input_2 (InputLayer)        [(None, 40)]                 0         []                            
                                                                                                  
 tf_roberta_model (TFRobert  TFBaseModelOutputWithPooli   1246456   ['input_1[0][0]',             
 aModel)                     ngAndCrossAttentions(last_   32         'input_2[0][0]']             
                             hidden_state=(None, 40, 76                                           
                             8),                                                              

## Model Training

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='/content/drive/MyDrive/MODEL2_RoBERTa/cp-{epoch:04d}.ckpt"',  # File path format to save the model
    save_freq='epoch',                      # Save the model after every epoch
    save_weights_only=True,                # Save the entire model (set to True to save only weights)
    verbose=1                               # Verbosity mode, 1 = show messages
)


# Train model
history = model.fit(
    [X_train_encoded['input_ids'], X_train_encoded['attention_mask']],
    train_labels,
    validation_data=(
      [X_val_encoded['input_ids'], X_val_encoded['attention_mask']], val_labels),
    batch_size=256,
    epochs=5,
    callbacks=[checkpoint],
    verbose=1
)

## Produce Submission File

In [ ]:
with open('/content/drive/MyDrive/test_cleaned.txt', 'r', encoding='utf-8') as file:
    test_content = file.readlines()

In [ ]:
print(test_content)

['sea doo pro sea scooter sad_emoji sport with the portable sea-doo seascootersave air stay longer in the water and\n', 'shuck well i work all week so now i can not come cheer you on ! oh and put those battery in your calculator ! ! !\n', 'i cant stay away from bug thats my baby\n', "no ma'am ! ! ! lol im perfectly fine and not contagious anymore lmao\n", 'whenever i fall asleep watching the tv i always wake up with a headache\n', 'he need to get rid of that thing ! it scare me lol but he do not need a car either he need driver ed again\n', 'it whatever in a terrible mood sad_emoji\n', 'yes ! rt thanks jordan i love you and i am going to call you later !\n', 'my friend text me to check up on me last night\n', 'please when will your come to europe and sweden ? ?\n', 'watch some of you all dumb ass get lock up today\n', 'obsessed with you killed it ! ! ! best album ever love yew roycee ! ! kiss_emoji rt me\n', 'robert de niro is not gay but with a name like lewy i would understand if you

In [ ]:
# Tokenize test data
X_test_encoded = tokenizer(test_content,
                            padding=True,
                            truncation=True,
                            max_length=max_len,
                            return_tensors='tf')

In [ ]:
print(X_test_encoded)

{'input_ids': <tf.Tensor: shape=(10000, 40), dtype=int32, numpy=
array([[    0, 16466,   109, ...,     1,     1,     1],
       [    0,  1193,  5858, ...,     1,     1,     1],
       [    0,   118, 17672, ...,     1,     1,     1],
       ...,
       [    0,  3654,   396, ..., 50118,     2,     1],
       [    0, 11990,  1531, ...,     1,     1,     1],
       [    0,  5349,    10, ...,     1,     1,     1]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(10000, 40), dtype=int32, numpy=
array([[1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0],
       ...,
       [1, 1, 1, ..., 1, 1, 0],
       [1, 1, 1, ..., 0, 0, 0],
       [1, 1, 1, ..., 0, 0, 0]], dtype=int32)>}


In [ ]:
from datetime import datetime
y_pred = model.predict([X_test_encoded['input_ids'], X_test_encoded['attention_mask']], verbose = 1, batch_size = 256)

if(len(y_pred) != 10000):
   print('Wrong size')
else:
   y_pred[y_pred <= 0.5] = -1
   y_pred[y_pred > 0.5] = 1
   y_pred = y_pred.astype(int)

   df = pd.read_csv('/content/drive/MyDrive/sample_submission.csv')
   df.Prediction = y_pred

   time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
   filename = "submission" + "_" + time + ".csv"
   store_path = '/content/drive/MyDrive/Submissions/' + filename
   df.to_csv(store_path, index = False)
   print("Submission stored")

40/40 [==============================] - 17s 335ms/step
Submission stored
